# KRIOS GIS — Fingrid Grid Capacity Layer Check

This notebook discovers and validates the **Fingrid substation consumption capacity** dataset — the authoritative source for the assignment's "Sähkön kulutuskapasiteetti" (electricity consumption capacity) requirement.

### Layer lineage

| Grid Scope web map | Assignment reference | ArcGIS FeatureServer |
|---|---|---|
| `Sähkön / kulutuskapasiteetti` (group) | "use Sähkön kulutuskapasiteetti layer" | — |
| `Substations` (active layer inside group) | The actual data layer | `Kytkinlaitokset_Fingrid` → `Sähköasemat_liityntäkapasiteetti` |

**Key field:** `Kulutus_25` = consumption headroom (MW) per substation

**Source:** [Grid Scope map](https://karttapalaute.fingrid.fi/?setlanguage=en&?link=3opMB) — Fingrid's interactive map for grid connection capacity estimates.

## Workflow

1. **Chunk 1** — List all FeatureServer services from the ArcGIS host; confirm `Kytkinlaitokset_Fingrid` exists.
2. **Chunk 2** — Drill into the layer, show its schema, field definitions, and a sample of attribute values.
3. **Chunk 3** — Fetch all 172 features with geometry, convert to GeoJSON, and visualize on an interactive map (GeoLibre).

In [3]:
import pandas as pd
import requests

# ArcGIS REST endpoint hosting Fingrid's geospatial data
BASE_URL = "https://services2.arcgis.com/uh3cDCipmuPcmxmx/ArcGIS/rest/services"

# Chunk 1: list all native ArcGIS services and keep FeatureServer items.
root_resp = requests.get(f"{BASE_URL}?f=pjson", timeout=45)
print("Root status:", root_resp.status_code)
root_resp.raise_for_status()
root_json = root_resp.json()

services = root_json.get("services", [])
services_df = pd.DataFrame(services)

if services_df.empty:
    raise RuntimeError("No services returned from ArcGIS root endpoint.")

services_df["service_url"] = (
    BASE_URL + "/" + services_df["name"].astype(str) + "/" + services_df["type"].astype(str)
    )
feature_services_df = services_df[services_df["type"].eq("FeatureServer")].copy()

print("Total services:", len(services_df))
print("FeatureServer count:", len(feature_services_df))
display(feature_services_df[["name", "type", "service_url"]].sort_values("name").reset_index(drop=True))

Root status: 200
Total services: 113
FeatureServer count: 113


,name,type,service_url
0,Asemakaavoitettualue_VESISTOALUEET,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
1,Asuinrakennus,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
2,Biokaasu,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
3,EVO_OYK,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
4,EVO_OYK_KIINTEISTÖTUNNUKSET,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
...,...,...,...
108,survey123_af37f13c356e4d2581a14b302ed42ec9_res...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
109,survey123_d8fbac0dec194234a9b66a751e1c8b8a_form,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
110,survey123_f9551175b74545e8a56f137d280efef9_fie...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
111,survey123_fa3c5b79a00e48b88cf0e75cbd493ac0_fie...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...


In [ ]:
# Chunk 2: extract the Fingrid substation connection capacity layer and inspect its schema + sample data.
# Assignment: "Grid capacity & headroom — Fingrid — use Sähkön kulutuskapasiteetti layer"
# Published as: Kytkinlaitokset_Fingrid -> Sähköasemat_liityntäkapasiteetti
# Key field: Kulutus_25 = consumption capacity headroom (MW)

TARGET_SERVICE_NAME = "Kytkinlaitokset_Fingrid"
TARGET_LAYER_HINT = "liityntäkapasiteetti"  # matches Sähköasemat_liityntäkapasiteetti

# Locate the target service
target_service = feature_services_df[feature_services_df["name"] == TARGET_SERVICE_NAME].copy()
if target_service.empty:
    raise RuntimeError(f"Target service not found: {TARGET_SERVICE_NAME}")

service_url = target_service.iloc[0]["service_url"]
service_resp = requests.get(f"{service_url}?f=pjson", timeout=45)
print("Service status:", service_resp.status_code)
service_resp.raise_for_status()
service_json = service_resp.json()

layers = service_json.get("layers", [])
layers_df = pd.DataFrame(layers)
if layers_df.empty:
    raise RuntimeError("No layers found inside target FeatureServer.")

# Find the substation connection capacity layer by name hint
layers_df["name_lc"] = layers_df["name"].astype(str).str.lower()
layer_match = layers_df[layers_df["name_lc"].str.contains(TARGET_LAYER_HINT.lower(), na=False)].copy()
selected_layer = layer_match.iloc[0] if not layer_match.empty else layers_df.iloc[0]

layer_url = f"{service_url}/{int(selected_layer['id'])}"
print("Selected layer:", selected_layer["name"])
print("Layer URL:", layer_url)

# Fetch layer metadata
meta_resp = requests.get(f"{layer_url}?f=pjson", timeout=45)
print("Layer metadata status:", meta_resp.status_code)
meta_resp.raise_for_status()
meta = meta_resp.json()

meta_summary = {
    "name": meta.get("name"),
    "type": meta.get("type"),
    "geometryType": meta.get("geometryType"),
    "objectIdField": meta.get("objectIdField"),
    "displayField": meta.get("displayField"),
    "maxRecordCount": meta.get("maxRecordCount"),
}
print("\nMetadata summary:", meta_summary)

# Show field definitions
fields_df = pd.DataFrame(meta.get("fields", []))
print("\nField count:", len(fields_df))
if not fields_df.empty:
    cols = [c for c in ["name", "alias", "type", "length", "nullable"] if c in fields_df.columns]
    display(fields_df[cols].sort_values("name").reset_index(drop=True))

# Query 10 sample records (no geometry, compact)
query_url = f"{layer_url}/query"
params = {
    "f": "json",
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "resultRecordCount": 10,
}
sample_resp = requests.get(query_url, params=params, timeout=45)
print("\nSample query status:", sample_resp.status_code)
sample_resp.raise_for_status()
sample_json = sample_resp.json()

rows = [f.get("attributes", {}) for f in sample_json.get("features", [])]
sample_df = pd.DataFrame(rows)
print("Sample row count:", len(sample_df))
if not sample_df.empty:
    preferred_cols = [c for c in ["SA", "Tyyppi", "Kulutus_25", "Tuotanto__", "Tuotanto_1", "Tuotanto_2", "X", "Y"] if c in sample_df.columns]
    display(sample_df[preferred_cols] if preferred_cols else sample_df.head(10))

In [ ]:
# Chunk 3: fetch all 172 features with geometry and visualise on an interactive GeoLibre map.
# Colours each substation by consumption capacity (Kulutus_25) — darker = more headroom.

import json
from pathlib import Path
import requests
from geolibre import Map

# Fetch all features with geometry in WGS84 (outSR=4326)
all_resp = requests.get(f"{layer_url}/query", params={
    "f": "json",
    "where": "1=1",
    "outFields": "SA,Tyyppi,Kulutus_25,Tuotanto__,Tuotanto_1,Tuotanto_2",
    "returnGeometry": "true",
    "outSR": 4326,
    "resultRecordCount": 2000,
}, timeout=60)
all_resp.raise_for_status()
raw = all_resp.json()

print(f"Fetched {len(raw.get('features', []))} features")

# Build a proper GeoJSON FeatureCollection
features = []
for feat in raw.get("features", []):
    geom = feat.get("geometry")
    attrs = feat.get("attributes", {})
    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [geom["x"], geom["y"]]
        },
        "properties": {
            "name": attrs.get("SA", ""),
            "voltage_kv": attrs.get("Tyyppi", ""),
            "consumption_mw": attrs.get("Kulutus_25", 0),
            "generation_1": attrs.get("Tuotanto__", 0),
            "generation_2": attrs.get("Tuotanto_1", 0),
            "generation_3": attrs.get("Tuotanto_2", 0),
        }
    })

fc = {"type": "FeatureCollection", "features": features}

# Save to a temp GeoJSON for GeoLibre to load
geojson_path = Path("../data/processed/fingrid_substations.geojson")
geojson_path.parent.mkdir(parents=True, exist_ok=True)
geojson_path.write_text(json.dumps(fc, ensure_ascii=False))
print(f"Saved {len(features)} features to {geojson_path}")

# Create an interactive GeoLibre map
# Center on Finland (approx centroid of data extent)
m = Map(
    center=[25.5, 64.5],  # [lng, lat]
    zoom=5,
    basemap="OpenFreeMap",
    layout="embed",
    height="700px",
)

# Add the substations as a graduated choropleth (colour by consumption_mw)
m.add_choropleth(
    str(geojson_path),
    column="consumption_mw",
    name="Fingrid Substations (consumption capacity MW)",
    class_count=5,
    colormap="YlOrRd",  # yellow → orange → red (lighter = less capacity)
    scheme="natural-breaks",
    # Circle markers with hover popups
    type="circle",
    radius=8,
    stroke=True,
    popup=["name", "voltage_kv", "consumption_mw"],
)

m